In [2]:
# BAN200 Group Project - Sentiment Analysis of Airline Reviews
# Steps 1-6 of our pipeline

# install and import what we need
!pip install nltk
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

import re
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# upload the dataset file when prompted
from google.colab import files
uploaded = files.upload()

# Step 1: load the dataset and explore it
df = pd.read_csv("Airline_review.csv")
print(df.shape)
print(df["Recommended"].value_counts())
print(df[["Review", "Recommended"]].isnull().sum())

# Step 2: turn Recommended into a label we can use (1 = yes, 0 = no)
df["label"] = df["Recommended"].map({"yes": 1, "no": 0})

# Step 3: tokenize the review text (lowercase + split into words)
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z']+", text)
    return tokens

df["tokens"] = df["Review"].apply(tokenize)

# Step 4: remove stop words and lemmatize
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_tokens(tokens):
    cleaned = [t for t in tokens if t not in stop_words and len(t) > 1]
    lemmatized = [lemmatizer.lemmatize(t) for t in cleaned]
    return lemmatized

df["clean_tokens"] = df["tokens"].apply(clean_tokens)

print(df["Review"][0][:200])
print(df["clean_tokens"][0][:20])

# Step 5: simple lexicon based classifier
# just a small list of positive/negative words for now, we can add more later
positive_words = {"good", "great", "excellent", "friendly", "comfortable", "helpful",
                   "smooth", "efficient", "clean", "nice", "amazing", "wonderful",
                   "professional", "recommend", "enjoyed", "polite", "attentive", "punctual"}

negative_words = {"bad", "poor", "terrible", "delayed", "rude", "uncomfortable",
                   "cancelled", "worst", "awful", "horrible", "disappointing", "late",
                   "lost", "broken", "dirty", "unhelpful", "never", "refund"}

def lexicon_score(tokens):
    score = 0
    for t in tokens:
        if t in positive_words:
            score += 1
        elif t in negative_words:
            score -= 1
    return score

df["lexicon_score"] = df["clean_tokens"].apply(lexicon_score)
df["lexicon_pred"] = (df["lexicon_score"] > 0).astype(int)

accuracy = (df["lexicon_pred"] == df["label"]).mean()
print("lexicon accuracy:", accuracy)

# Step 6: bag of words, split by class, to see top words
pos_reviews = df[df["label"] == 1]["clean_tokens"]
neg_reviews = df[df["label"] == 0]["clean_tokens"]

pos_counts = Counter(word for tokens in pos_reviews for word in tokens)
neg_counts = Counter(word for tokens in neg_reviews for word in tokens)

print("top words in positive reviews:", pos_counts.most_common(15))
print("top words in negative reviews:", neg_counts.most_common(15))

# save cleaned data so we can use it for Naive Bayes later
df[["Review", "clean_tokens", "label"]].to_csv("cleaned_reviews.csv", index=False)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Saving Airline_review.csv to Airline_review.csv
(23171, 20)
Recommended
no     15364
yes     7807
Name: count, dtype: int64
Review         0
Recommended    0
dtype: int64
  Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes 
['moroni', 'moheli', 'turned', 'pretty', 'decent', 'airline', 'online', 'booking', 'worked', 'well', 'checkin', 'boarding', 'fine', 'plane', 'looked', 'well', 'maintained', 'short', 'flight', 'minute']
lexicon accuracy: 0.8527469681929999
top words in positive reviews: [('flight', 14980), ('seat', 5333), ('time', 5225), ('good', 4766), ('service', 4667), ('airline', 4641), ('crew', 3933), ('food', 3102), ('cabin', 3075), ('staff', 3004), ('check', 2830), ('plane', 2435), ('friendly', 2291), ('hour', 2241), ('would', 2142)]
top words in negative reviews: [('flight', 35788), ('airline', 14200), ('hour', 10900), ('time', 10